In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import joblib
import os

### Load Dataset

You can use a cybersecurity dataset like:

UNSW-NB15

In [2]:
df_train = pd.read_csv("../data/UNSW_NB15_training-set.csv")

df_test = pd.read_csv("../data/UNSW_NB15_testing-set.csv")

df_train.head()

df_test.head()

df_train.shape

df_test.shape

(82332, 45)

#### Inspect Data

In [3]:
df_train.info()
df_test.info()

df_train["label"].value_counts()
df_test["label"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  object 
 3   service            175341 non-null  object 
 4   state              175341 non-null  object 
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  si

label
1    45332
0    37000
Name: count, dtype: int64

### Feature / Target Split / Train Test Split

In [4]:
X_train = df_train.drop("label", axis=1)
y_train = df_train["label"]
X_test = df_test.drop("label", axis=1)
y_test = df_test["label"]

### Feature Scaling

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(exclude=['object']).columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [10]:
'''scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)'''

'scaler = StandardScaler()\n\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)'

#### Train ML Model

In [11]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train_processed, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200, random_state=42)

### Model Predictions

In [12]:
predictions = model.predict(X_test_processed)
probabilities = model.predict_proba(X_test_processed)[:,1]

### Evaluate Model

In [13]:
print(classification_report(y_test, predictions))

print(confusion_matrix(y_test, predictions))

roc = roc_auc_score(y_test, probabilities)
print("ROC AUC:", roc)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     37000
           1       1.00      1.00      1.00     45332

    accuracy                           1.00     82332
   macro avg       1.00      1.00      1.00     82332
weighted avg       1.00      1.00      1.00     82332

[[36973    27]
 [    2 45330]]
ROC AUC: 0.9999999421684105


### Save Model for the Agent

In [14]:
os.makedirs("../models", exist_ok=True)

In [15]:
joblib.dump(model, "../models/threat_model.pkl")

['../models/threat_model.pkl']

In [16]:
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']